[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/13_DeepTab/46_deeptab_tabular_deep_learning.ipynb)

> 📎 **Optional module — reference style.** Module 13 is optional. Like the appendices, this notebook is written as a demo / reference: it focuses on *seeing* a library at work rather than interactive exercises. It is built to run end-to-end *without* the optional library — it falls back to a small built-in stand-in — so you can read and run it offline. Install the optional library (see **Install** below) to swap the stand-in for the real thing.

---

# 📓 Notebook 46 — DeepTab: Deep Learning for Tabular Data

> **Module:** 13 · Optional · **Type:** Reference · **Estimated time:** 60–90 min · **Difficulty:** Intermediate → Advanced

For two decades, structured/tabular data — the spreadsheets, transaction logs, and feature stores that run most of a business — has belonged to **gradient-boosted trees** (XGBoost, LightGBM, CatBoost). On the median tabular benchmark they are still the thing to beat: fast, robust, almost hyperparameter-forgiving.

But the gap has narrowed. A wave of architectures designed *specifically for tables* — **FT-Transformer**, **SAINT**, **NODE**, **TabM**, **ResNet**, and sequence models such as **Mamba** — now match or beat boosting on large, complex datasets, and they unlock things trees simply cannot do natively:

- **Distributional regression** — predict an entire probability distribution (mean *and* variance, or a Poisson/Gamma), not just a point estimate. Gold for pricing, risk, and inventory.
- **Latent embeddings** — learn dense vector representations of each row that you can reuse downstream (similarity search, clustering, feeding another model).
- **End-to-end multimodal** — fuse tabular columns with text or image towers in a single differentiable model.
- **Transfer learning** — pre-train on one table, fine-tune on another.

**[DeepTab](https://github.com/OpenTabular/DeepTab)** (from the OpenTabular project) wraps roughly **18** of these models behind a single, clean scikit-learn `BaseEstimator` API. Every architecture ships as a matching `<Name>Classifier`, `<Name>Regressor`, and `<Name>LSS` (distributional) trio — so switching from a Transformer to Mamba to a plain MLP is a one-line change of class name.

## 🎯 Learning objectives

By the end you will be able to:

1. Decide **when** a tabular deep-learning model is worth it versus reaching for a gradient-boosted tree (spoiler: try the trees first).
2. Train a **classifier** and a **regressor** with DeepTab's sklearn-style `.fit` / `.predict` / `.predict_proba` API, and swap architectures by name.
3. Use **distributional regression (LSS)** to predict full distributions and reason about uncertainty for risk and pricing.
4. Extract **latent embeddings** with `.encode()` and feed them to a downstream model.
5. **Tune** hyperparameters with both DeepTab's built-in `optimize_hparams` and sklearn's `RandomizedSearchCV`.

## ✅ Prerequisites

- **NB 14 — scikit-learn basics**: the `fit` / `predict` estimator pattern, train/test splits, metrics.
- **NB 16 — feature engineering**: numeric vs categorical columns, encoding, why preprocessing matters.
- **NB 04 appendix A1–A3 — PyTorch foundations**: tensors, the training loop, epochs and learning rates (DeepTab is built on PyTorch + Lightning, so these terms recur).

## 📦 Install

```bash
pip install deeptab
```

> ⚠️ **Heads-up:** `deeptab` pulls in **PyTorch** and **PyTorch Lightning**, which are large downloads (hundreds of MB) and may need a CUDA-matched build for GPU. That is exactly why this notebook is **offline-first**: if `deeptab` is not installed, every cell falls back to a lightweight scikit-learn stand-in so you can read and run the whole thing right now.

## 🧰 Setup & imports

Standard scientific Python stack. Nothing here is downloaded — the dataset is synthetic and generated inline.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    mean_absolute_error, r2_score,
)
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.ensemble import (
    HistGradientBoostingClassifier, HistGradientBoostingRegressor,
)
from sklearn.preprocessing import OrdinalEncoder
from sklearn.base import BaseEstimator

from scipy.stats import randint, uniform, norm

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 30)
print('Imports OK — versions:')
print('  numpy   ', np.__version__)
print('  pandas  ', pd.__version__)

## 🔌 Smoke test: is DeepTab installed?

This is the gate the whole notebook hangs on. We try to import `deeptab` and set a single flag, `HAS_DEEPTAB`. Every later cell branches on it:

- `HAS_DEEPTAB == True` → use the **real** DeepTab models.
- `HAS_DEEPTAB == False` → use a **scikit-learn stand-in** that mimics the same `.fit` / `.predict` / `.predict_proba` / `.encode` calls.

The stand-in is *not* a DeepTab implementation — it is a teaching scaffold so the API shape and the surrounding analysis still run offline.

In [ ]:
try:
    import deeptab
    from deeptab.models import (
        MambularClassifier, MambularRegressor, MambularLSS,
    )
    HAS_DEEPTAB = True
    print(f'✅ DeepTab {getattr(deeptab, "__version__", "?")} found — using real models.')
except Exception as exc:  # ImportError or any load-time failure
    HAS_DEEPTAB = False
    print('ℹ️  DeepTab not installed — running with the offline sklearn stand-in.')
    print(f'   (import said: {type(exc).__name__})')
    print('   Install with:  pip install deeptab')

### The offline stand-in

When DeepTab is missing we define drop-in classes with the **same method signatures** as DeepTab's estimators. Internally they are ordinary scikit-learn models:

- classifier / regressor → an `MLPClassifier` / `MLPRegressor` (a real, if small, neural net) sitting behind an ordinal encoder for the categorical columns;
- `.encode(X)` → returns the last hidden-layer activations as a cheap 'embedding';
- `MambularLSS` → fits the mean with an MLP and estimates a constant residual standard deviation, then exposes `predict_proba`-style distributional outputs.

They accept (and harmlessly ignore) DeepTab-only keyword arguments such as `d_model`, `n_layers`, `max_epochs`, `lr`, `patience`, `family`, and `numerical_preprocessing` so that the *exact same calling code* works in both modes. Read the bodies once; after that, treat them as DeepTab.

In [ ]:
# A tiny helper: ordinal-encode object columns so plain sklearn nets accept them.
def _to_numeric_matrix(X):
    X = X.copy()
    cat_cols = X.select_dtypes(include=['object', 'category']).columns
    if len(cat_cols):
        enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        X[cat_cols] = enc.fit_transform(X[cat_cols].astype(str))
    return X.to_numpy(dtype=float)


class _StandInBase(BaseEstimator):
    '''Mimics DeepTab's estimator surface; ignores DL-only kwargs.'''
    # NOTE: explicit (no **kwargs) so sklearn get_params/set_params and
    # RandomizedSearchCV can introspect and clone these estimators.
    def __init__(self, d_model=64, n_layers=4, dropout=0.1,
                 numerical_preprocessing='ple', n_bins=50):
        self.d_model = d_model
        self.n_layers = n_layers
        self.dropout = dropout
        self.numerical_preprocessing = numerical_preprocessing
        self.n_bins = n_bins

    def _hidden(self):
        # map 'depth' onto an MLP hidden geometry, capped for speed offline
        width = min(self.d_model, 64)
        depth = min(self.n_layers, 3)
        return tuple([width] * max(depth, 1))

    def _build_net(self):
        raise NotImplementedError

    def fit(self, X, y, max_epochs=100, lr=1e-3, patience=10, **kw):
        self.net_ = self._build_net()   # built here so set_params is respected
        self._Xnum = _to_numeric_matrix(X)
        self.net_.fit(self._Xnum, y)
        return self

    def predict(self, X):
        return self.net_.predict(_to_numeric_matrix(X))

    def encode(self, X):
        '''Last-hidden-layer activations as a stand-in embedding.'''
        Z = _to_numeric_matrix(X)
        # forward-propagate manually through the fitted MLP up to last hidden layer
        a = Z
        for w, b in zip(self.net_.coefs_[:-1], self.net_.intercepts_[:-1]):
            a = np.maximum(0.0, a @ w + b)  # relu
        return a

    def optimize_hparams(self, X, y, **kw):
        print('   [stand-in] optimize_hparams is a no-op; returning self.')
        return self.fit(X, y)


class StandInClassifier(_StandInBase):
    def _build_net(self):
        return MLPClassifier(hidden_layer_sizes=self._hidden(),
                             max_iter=300, random_state=RANDOM_STATE)
    def predict_proba(self, X):
        return self.net_.predict_proba(_to_numeric_matrix(X))


class StandInRegressor(_StandInBase):
    def _build_net(self):
        return MLPRegressor(hidden_layer_sizes=self._hidden(),
                            max_iter=400, random_state=RANDOM_STATE)


class StandInLSS(_StandInBase):
    '''Distributional stand-in: MLP mean + constant residual sigma.'''
    def __init__(self, family='normal', d_model=64, n_layers=4, dropout=0.1,
                 numerical_preprocessing='ple', n_bins=50):
        super().__init__(d_model, n_layers, dropout,
                         numerical_preprocessing, n_bins)
        self.family = family
    def _build_net(self):
        return MLPRegressor(hidden_layer_sizes=self._hidden(),
                            max_iter=400, random_state=RANDOM_STATE)
    def fit(self, X, y, max_epochs=100, lr=1e-3, patience=10, family=None, **kw):
        if family is not None:
            self.family = family
        self.net_ = self._build_net()
        self._Xnum = _to_numeric_matrix(X)
        self.net_.fit(self._Xnum, y)
        resid = y - self.net_.predict(self._Xnum)
        self.sigma_ = float(np.std(resid)) or 1.0
        return self
    def predict(self, X):
        '''Return (mean, std) per row, mirroring a distributional head.'''
        mu = self.net_.predict(_to_numeric_matrix(X))
        sd = np.full_like(mu, self.sigma_)
        return np.column_stack([mu, sd])

print('Stand-in classes defined.')

### One factory, two backends

To keep later cells clean we wrap the choice in small factory functions. `make_classifier()` returns a real `MambularClassifier` when DeepTab is present, otherwise a `StandInClassifier`. Because both honour the same keyword arguments and methods, **the rest of the notebook never needs another `if HAS_DEEPTAB`** for basic fit/predict — it just calls the factory.

> 🔁 **Swapping architectures.** With real DeepTab, every model below has siblings — `FTTransformerClassifier`, `TabTransformerClassifier`, `ResNetClassifier`, `MLPClassifier`, `SAINTClassifier`, `TabMClassifier`, `NODEClassifier`, … — that take the same arguments. To try a Transformer instead of Mamba you change exactly one word: the class name.

In [ ]:
def make_classifier(**kw):
    if HAS_DEEPTAB:
        return MambularClassifier(**kw)
    return StandInClassifier(**kw)

def make_regressor(**kw):
    if HAS_DEEPTAB:
        return MambularRegressor(**kw)
    return StandInRegressor(**kw)

def make_lss(**kw):
    if HAS_DEEPTAB:
        return MambularLSS(**kw)
    return StandInLSS(**kw)

backend = 'real DeepTab' if HAS_DEEPTAB else 'sklearn stand-in'
print(f'Model factory ready — backend: {backend}.')

## 1️⃣ The value proposition — and when *not* to use it

Let's be honest, because hype helps no one ship reliable models.

### Start with gradient boosting

For a typical business table — tens of thousands of rows, a few dozen columns, a clear target — **XGBoost / LightGBM / `HistGradientBoosting` is your first call.** It trains in seconds, tolerates messy mixed-type data, rarely overfits catastrophically, and is hard to beat on accuracy per unit of effort. Deep learning for tables earns its keep only when one of the following is true:

| Reach for tabular DL when… | Because… |
|---|---|
| **You have a lot of data** (≫100k rows) | Neural nets keep improving with scale where trees plateau. |
| **You need a full distribution**, not a point | LSS heads give calibrated mean + variance for pricing, risk, demand. |
| **The problem is multimodal** (table + text/image) | One differentiable model can fuse towers end-to-end. |
| **You want reusable embeddings** | `.encode()` yields dense row vectors for search, clustering, or transfer. |
| **Transfer / pre-training helps** | Pre-train on a big related table, fine-tune on the small target one. |

If none of those apply, a boosted tree is probably the right — and cheaper — answer. The professional move is to **benchmark the tree baseline first** and only adopt DL if it clears that bar by a margin worth the extra complexity, compute, and GPU dependency. We keep a `HistGradientBoosting` baseline in this notebook for exactly that reason.

## 2️⃣ A synthetic business dataset (churn)

We build a small **customer-churn** frame inline — no downloads. It mixes **numeric** columns (tenure, monthly charges, support tickets, usage) with **categorical** ones (contract type, payment method, region). This mirrors the kind of mixed-type table DeepTab is designed for, and lets us show that its **PreTab** preprocessing detects column types straight from a pandas `DataFrame`.

~400 rows keeps everything fast — including real DeepTab on a CPU.

In [ ]:
N = 400
# Numeric signal from make_classification, then dress it up as a churn frame.
Xc, yc = make_classification(
    n_samples=N, n_features=6, n_informative=4, n_redundant=1,
    n_classes=2, weights=[0.7, 0.3], class_sep=1.2, random_state=RANDOM_STATE,
)

df = pd.DataFrame({
    'tenure_months':   (np.abs(Xc[:, 0]) * 18 + 6).round().astype(int),
    'monthly_charges': (np.abs(Xc[:, 1]) * 25 + 40).round(2),
    'support_tickets': np.clip((Xc[:, 2] * 2 + 2).round(), 0, None).astype(int),
    'avg_usage_gb':    (np.abs(Xc[:, 3]) * 30 + 10).round(1),
    'contract': rng.choice(['month-to-month', 'one-year', 'two-year'],
                           size=N, p=[0.55, 0.30, 0.15]),
    'payment':  rng.choice(['card', 'bank', 'e-check'], size=N),
    'region':   rng.choice(['north', 'south', 'east', 'west'], size=N),
})
# Make 'contract' genuinely informative: month-to-month churns more.
contract_risk = df['contract'].map({'month-to-month': 0.9,
                                     'one-year': 0.0, 'two-year': -0.9}).to_numpy()
churn_logit = Xc[:, :4].sum(axis=1) * 0.5 + contract_risk + rng.normal(0, 0.4, N)
df['churn'] = (churn_logit > np.median(churn_logit) - 0.1).astype(int)

print(f'Shape: {df.shape} | churn rate: {df.churn.mean():.1%}')
df.head()

In [ ]:
# Quick look at the type split PreTab would detect automatically.
num_cols = df.drop(columns='churn').select_dtypes('number').columns.tolist()
cat_cols = df.drop(columns='churn').select_dtypes('object').columns.tolist()
print('Numeric   :', num_cols)
print('Categorical:', cat_cols)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
sns.histplot(data=df, x='tenure_months', hue='churn', bins=20,
             ax=axes[0], multiple='stack')
axes[0].set_title('Tenure by churn')
sns.countplot(data=df, x='contract', hue='churn', ax=axes[1])
axes[1].set_title('Churn by contract type')
plt.tight_layout(); plt.show()

In [ ]:
feature_cols = num_cols + cat_cols
X = df[feature_cols].copy()       # keep it a DataFrame — DeepTab wants that
y = df['churn'].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE,
)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print(f'X dtype check — still a DataFrame? {isinstance(X_train, pd.DataFrame)}')

## 3️⃣ Classification with `MambularClassifier`

Now the headline. DeepTab's API is deliberately scikit-learn shaped, so if you know `.fit` / `.predict` / `.predict_proba`, you already know this.

```python
from deeptab.models import MambularClassifier

model = MambularClassifier(
    d_model=64,                    # width of the internal representation
    n_layers=4,                    # depth (here, Mamba blocks)
    numerical_preprocessing='ple', # piecewise-linear encoding of numerics
    n_bins=50,                     # bins for the PLE encoder
)
model.fit(X_train, y_train, max_epochs=150, lr=1e-4, patience=10)
preds = model.predict(X_test)
proba = model.predict_proba(X_test)
```

Key arguments worth knowing:

- **`d_model`** — the model's hidden width. Bigger = more capacity.
- **`n_layers`** — depth. For Mamba these are state-space blocks; for `FTTransformer` they'd be attention layers.
- **`numerical_preprocessing`** — how numeric columns are encoded. `'ple'` (piecewise-linear encoding) bins each feature and learns per-bin slopes — a strong default that often beats plain `'standardization'`; `'box-cox'` is also available.
- **`max_epochs` / `lr` / `patience`** — standard training-loop knobs passed to `.fit`; `patience` drives early stopping.

Offline, the call below runs the stand-in with shorter training for speed.

In [ ]:
# Lighter settings keep the offline run snappy; bump these up with real DeepTab.
EPOCHS = 150 if HAS_DEEPTAB else 5

clf = make_classifier(d_model=64, n_layers=4,
                      numerical_preprocessing='ple', n_bins=50)
clf.fit(X_train, y_train, max_epochs=EPOCHS, lr=1e-4, patience=10)

pred = clf.predict(X_test)
proba = clf.predict_proba(X_test)[:, 1]

print(f'Backend : {backend}')
print(f'Accuracy: {accuracy_score(y_test, pred):.3f}')
print(f'ROC-AUC : {roc_auc_score(y_test, proba):.3f}')
print()
print(classification_report(y_test, pred, target_names=['stay', 'churn']))

### Honest baseline: the gradient-boosted tree

As promised, we always check the cheap, strong baseline. On a small clean table like this, do not be surprised if the tree wins — that is the whole point of the GBMs-first rule. The deep model's advantages (distributions, embeddings, scale) show up *elsewhere*, not necessarily on accuracy here.

In [ ]:
# HistGradientBoosting handles categoricals natively if we encode to ints first.
X_tr_enc = _to_numeric_matrix(X_train)
X_te_enc = _to_numeric_matrix(X_test)

gbm = HistGradientBoostingClassifier(random_state=RANDOM_STATE)
gbm.fit(X_tr_enc, y_train)
gbm_pred = gbm.predict(X_te_enc)
gbm_proba = gbm.predict_proba(X_te_enc)[:, 1]

print('--- Tree baseline (HistGradientBoosting) ---')
print(f'Accuracy: {accuracy_score(y_test, gbm_pred):.3f}')
print(f'ROC-AUC : {roc_auc_score(y_test, gbm_proba):.3f}')

print('\nRule of thumb: adopt the deep model only if it clears this bar by a')
print('margin worth the extra compute, GPU dependency, and tuning effort.')

### Swapping the architecture

This is DeepTab's signature convenience. With the real library installed, every line below would train a *different* deep architecture on the same `X_train, y_train` — same arguments, same `.fit` / `.predict`:

```python
from deeptab.models import (
    FTTransformerClassifier, TabTransformerClassifier, ResNetClassifier,
    MLPClassifier, SAINTClassifier, TabMClassifier, NODEClassifier,
)

model = FTTransformerClassifier(d_model=64, n_layers=4)   # attention over features
model = SAINTClassifier(d_model=64, n_layers=4)           # row + column attention
model = TabMClassifier(d_model=64, n_layers=4)            # efficient MLP ensemble
model = NODEClassifier(n_layers=4)                        # neural oblivious trees
model.fit(X_train, y_train, max_epochs=150, lr=1e-4)
```

No new preprocessing, no new training code — just the class name. That makes **architecture search a loop over class names**, which we lean on when tuning.

## 4️⃣ Regression with `MambularRegressor`

Same API, continuous target. Let's predict a customer's **expected monthly revenue** (a pricing/forecasting flavour). We synthesise a regression target from the same feature space.

```python
from deeptab.models import MambularRegressor
reg = MambularRegressor(d_model=64, n_layers=4)
reg.fit(X_train, y_train, max_epochs=100, lr=1e-4)
y_hat = reg.predict(X_test)
```

In [ ]:
# Build a continuous target: revenue driven by charges, usage, tenure + noise.
rev = (df['monthly_charges'] * 1.4
       + df['avg_usage_gb'] * 0.6
       + df['tenure_months'] * 0.3
       + rng.normal(0, 6, len(df)))
df['revenue'] = rev.round(2)

yr = df['revenue'].to_numpy()
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X, yr, test_size=0.25, random_state=RANDOM_STATE)

reg = make_regressor(d_model=64, n_layers=4)
reg.fit(Xr_train, yr_train, max_epochs=EPOCHS, lr=1e-4)
yhat = reg.predict(Xr_test)

print(f'Backend: {backend}')
print(f'MAE : {mean_absolute_error(yr_test, yhat):.2f}')
print(f'R²  : {r2_score(yr_test, yhat):.3f}')

plt.figure(figsize=(5, 5))
plt.scatter(yr_test, yhat, alpha=0.6, edgecolor='k', linewidth=0.3)
lims = [min(yr_test.min(), yhat.min()), max(yr_test.max(), yhat.max())]
plt.plot(lims, lims, 'r--', label='perfect')
plt.xlabel('actual revenue'); plt.ylabel('predicted revenue')
plt.title('Regression: predicted vs actual'); plt.legend()
plt.tight_layout(); plt.show()

## 5️⃣ Distributional regression (LSS) with `MambularLSS`

This is where deep tabular models do something trees can't do out of the box. **LSS = Location, Scale, Shape.** Instead of a single point prediction, the model outputs the *parameters of a probability distribution* for each row — so you get **uncertainty for free**.

```python
from deeptab.models import MambularLSS
lss = MambularLSS(d_model=64, n_layers=8, dropout=0.2)
lss.fit(X_train, y_train, max_epochs=150, lr=1e-4, patience=10, family='normal')
```

The **`family`** argument picks the distribution: `'normal'`, `'poisson'` (counts), `'gamma'` (positive, skewed — claims, durations), `'beta'` (rates in 0–1), `'studentt'` (heavy tails), and more.

### Why a distribution beats a point estimate

A point forecast of '€120 revenue' hides whether the model means *€120 ± 3* or *€120 ± 60*. For real decisions the spread is the whole game:

- **Pricing / risk** — set a margin that covers the *downside tail*, not just the average.
- **Inventory / demand** — stock to a high quantile so you rarely run out; the distribution gives you that quantile directly.
- **Triage** — flag the rows where the model is *uncertain* for human review.

> 🔗 **Complementary tool — conformal prediction (NB A5).** LSS gives a *parametric* uncertainty estimate (it assumes the family is right). **Conformal prediction** wraps *any* model to produce intervals with a *distribution-free finite-sample coverage guarantee*. They pair well: use LSS for a rich shape, conformal to certify calibrated coverage.

In [ ]:
lss = make_lss(d_model=64, n_layers=8, dropout=0.2)
lss.fit(Xr_train, yr_train, max_epochs=EPOCHS, lr=1e-4, patience=10, family='normal')

# Real DeepTab returns distribution parameters; our stand-in returns (mean, std).
# We normalise both to a (mu, sigma) view for the demo.
if HAS_DEEPTAB:
    # DeepTab LSS .predict yields per-row distribution params; shapes vary by
    # family/version. We defensively coerce to a 2-column (mu, sigma) array.
    raw = np.asarray(lss.predict(Xr_test), dtype=float)
    raw = raw.reshape(len(Xr_test), -1)
    mu = raw[:, 0]
    sigma = np.abs(raw[:, 1]) if raw.shape[1] > 1 else np.full(len(mu), raw[:, 0].std())
else:
    params = lss.predict(Xr_test)
    mu, sigma = params[:, 0], params[:, 1]

print('First 5 rows — predicted distribution (normal):')
for i in range(5):
    lo, hi = mu[i] - 1.96 * sigma[i], mu[i] + 1.96 * sigma[i]
    print(f'  row {i}: mean={mu[i]:7.2f}  sd={sigma[i]:6.2f}  95% interval=[{lo:7.2f}, {hi:7.2f}]')

In [ ]:
# Visualise the predicted distribution for a few customers.
fig, ax = plt.subplots(figsize=(8, 4))
grid = np.linspace(mu.min() - 3 * sigma.max(), mu.max() + 3 * sigma.max(), 300)
for i in range(4):
    pdf = norm.pdf(grid, loc=mu[i], scale=max(sigma[i], 1e-6))
    ax.plot(grid, pdf, label=f'customer {i} (μ={mu[i]:.0f})')
    ax.axvline(yr_test[i], color=ax.lines[-1].get_color(), ls=':', alpha=0.6)
ax.set_title('Predicted revenue distributions (dotted = actual)')
ax.set_xlabel('revenue'); ax.set_ylabel('density'); ax.legend()
plt.tight_layout(); plt.show()

print('Decision example — stock/price to the 10th percentile (pessimistic):')
q10 = mu - 1.2816 * sigma
print(f'  mean of point forecasts : {mu.mean():.2f}')
print(f'  mean of 10th-pct forecasts: {q10.mean():.2f}  <- the cautious number')

## 6️⃣ Latent embeddings via `model.encode(X)`

A trained deep tabular model has learned, in its hidden layers, a dense vector representation of each row. `model.encode(X)` exposes it:

```python
latent = model.encode(X_test)   # shape (n_rows, d_model-ish)
```

What is it good for?

- **Downstream models** — feed the embedding to a simple, fast model (logistic regression, k-NN) instead of the raw columns.
- **Similarity / retrieval** — nearest neighbours in embedding space = 'customers like this one'.
- **Clustering & visualisation** — segment the embedding space.
- **Transfer** — embeddings from a model trained on a big table can seed a smaller task.

Below we encode the churn data and train a plain logistic regression on the embeddings — a cheap way to reuse the deep model's learned features.

In [ ]:
from sklearn.linear_model import LogisticRegression

Z_train = np.asarray(clf.encode(X_train), dtype=float)
Z_test  = np.asarray(clf.encode(X_test), dtype=float)
print(f'Embedding shape — train: {Z_train.shape}, test: {Z_test.shape}')

# Downstream: a tiny linear model on top of the learned representation.
head = LogisticRegression(max_iter=1000)
head.fit(Z_train, y_train)
head_acc = accuracy_score(y_test, head.predict(Z_test))
print(f'Logistic regression on embeddings — accuracy: {head_acc:.3f}')

# Visualise the embedding in 2D (PCA) coloured by churn.
from sklearn.decomposition import PCA
if Z_test.shape[1] >= 2:
    emb2d = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(Z_test)
    plt.figure(figsize=(5.5, 4.5))
    sc = plt.scatter(emb2d[:, 0], emb2d[:, 1], c=y_test, cmap='coolwarm',
                     alpha=0.7, edgecolor='k', linewidth=0.3)
    plt.title('Row embeddings (PCA) coloured by churn')
    plt.xlabel('PC1'); plt.ylabel('PC2'); plt.colorbar(sc, label='churn')
    plt.tight_layout(); plt.show()
else:
    print('Embedding too narrow to plot in 2D (stand-in geometry).')

## 7️⃣ Hyperparameter tuning

Two complementary routes.

### a) Built-in Bayesian search — `model.optimize_hparams`

DeepTab ships its own tuner that knows each architecture's sensible search space and uses Bayesian optimisation:

```python
model = MambularClassifier()
best = model.optimize_hparams(X_train, y_train)   # returns a tuned estimator
```

This is the quickest way to a strong model — no search space to hand-design.

### b) sklearn `RandomizedSearchCV` — full ecosystem integration

Because DeepTab models are real `BaseEstimator`s, every sklearn tuning tool works, with cross-validation and any scorer:

```python
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_dist = {
    'd_model': randint(32, 128),
    'n_layers': randint(2, 10),
    'numerical_preprocessing': ['ple', 'standardization'],
}
rs = RandomizedSearchCV(MambularClassifier(), param_dist,
                        n_iter=10, cv=3, scoring='accuracy')
rs.fit(X_train, y_train, max_epochs=5, rebuild=False)
```

> The `rebuild=False` fit kwarg tells DeepTab to reuse the preprocessing pipeline across CV folds instead of rebuilding it each time — a big speed-up during search. (`max_epochs=5` keeps each fit short while searching.)

In [ ]:
# (a) Built-in tuner — real on DeepTab, a no-op refit on the stand-in.
tuned = make_classifier(d_model=64, n_layers=4)
if HAS_DEEPTAB:
    tuned = tuned.optimize_hparams(X_train, y_train)
    print('optimize_hparams returned a tuned model.')
else:
    tuned.optimize_hparams(X_train, y_train)
    print('[stand-in] optimize_hparams refit the model (no real search).')
print(f'Tuned accuracy: {accuracy_score(y_test, tuned.predict(X_test)):.3f}')

In [ ]:
# (b) RandomizedSearchCV — works identically on both backends.
param_dist = {
    'd_model': randint(32, 128),
    'n_layers': randint(2, 6),
    'numerical_preprocessing': ['ple', 'standardization'],
}
base = make_classifier()

# fit kwargs differ: real DeepTab accepts max_epochs/rebuild; the stand-in
# ignores unknown kwargs gracefully, so we pass them either way.
fit_params = {'max_epochs': EPOCHS, 'rebuild': False} if HAS_DEEPTAB else {}

rs = RandomizedSearchCV(base, param_dist, n_iter=4, cv=3,
                        scoring='accuracy', random_state=RANDOM_STATE)
rs.fit(X_train, y_train, **fit_params)

print(f'Best CV accuracy : {rs.best_score_:.3f}')
print(f'Best params      : {rs.best_params_}')
print(f'Test accuracy    : {accuracy_score(y_test, rs.predict(X_test)):.3f}')

## 🧪 Exercises

These are best done with **real DeepTab installed** (`pip install deeptab`), but the offline stand-in lets you draft and run the *code shape* first.

1. **Architecture bake-off.** Loop over `['Mambular', 'FTTransformer', 'ResNet', 'TabM']`, train each classifier on the churn data, and tabulate accuracy + ROC-AUC + train time. Does any deep model beat the `HistGradientBoosting` baseline here? Why might it not, on 400 rows?
2. **Bigger data.** Regenerate the dataset with `N = 50_000`. Re-run the deep model vs the tree. Does the gap change? This is the 'DL likes scale' lesson in miniature.
3. **Pick the right family.** Build a *count* target (e.g. number of support tickets next month) and fit `MambularLSS(family='poisson')`. Compare the predicted intervals to a `family='normal'` fit. Which respects 'counts can't be negative'?
4. **Embeddings for retrieval.** Use `.encode()` plus `sklearn`'s `NearestNeighbors` to build a 'find me 5 similar customers' function.
5. **Tune for real.** Run `RandomizedSearchCV` with `scoring='roc_auc'` and a wider `n_layers` range. Compare its winner to `optimize_hparams`.
6. **Conformal pairing (stretch).** Revisit **NB A5** and wrap your trained regressor in a split-conformal procedure. Compare conformal interval width to the LSS `±1.96σ` interval — which is wider, and which has a coverage *guarantee*?

## 🧠 Key takeaways

- **Trees first.** Gradient boosting (XGBoost/LightGBM/HistGB) is the right default for most tabular problems. Benchmark it before adopting deep learning.
- **DeepTab = sklearn API over ~18 tabular DL architectures.** Every model is a `<Name>Classifier` / `<Name>Regressor` / `<Name>LSS`; swap architectures by changing one class name.
- **Where DL earns its place:** large data, multimodal inputs, distributional outputs, reusable embeddings, transfer learning.
- **LSS (distributional regression)** predicts a whole distribution — mean *and* uncertainty — which is what risk, pricing, and inventory decisions actually need. Pair it with **conformal prediction (NB A5)** for guaranteed coverage.
- **`.encode()`** turns the model into a feature extractor: dense row embeddings for downstream models, similarity search, and clustering.
- **Tuning** is easy two ways: built-in `optimize_hparams` (Bayesian, zero-config) or any sklearn searcher like `RandomizedSearchCV`.
- **`numerical_preprocessing='ple'`** (piecewise-linear encoding) is a strong default for numeric columns; PreTab auto-detects numeric vs categorical straight from your DataFrame.
- This notebook ran **offline** on a stand-in — the *API and reasoning* transfer directly to the real library once installed.


## 🚀 Next step

You've reached the end of the optional Module 13 — and, with it, the modelled tour of the course's tabular and deep-learning toolkit.

- **Install it for real:** `pip install deeptab`, then re-run this notebook. Watch the stand-in messages disappear and the genuine Mamba / Transformer models take over.
- **Compare back to the core ML modules** — NB 14 (sklearn), NB 16 (feature engineering), NB A5 (conformal prediction) — to keep the GBMs-first instinct sharp.
- **Return to the course [README](../README.md)** for the full map, the capstones (Module 07), and what to build next.

DeepTab is one good entry point into tabular deep learning; the judgement you practised here — *try the simple strong baseline, reach for DL when it buys you something specific* — is the part that lasts. 🎓